In [ ]:
from pathlib import Path
from collections import defaultdict
import math

import pandas as pd
import torch
from torch_geometric.data import Data
from rdkit import Chem
from rdkit.Chem import MolFromMol2File

# ===== 路径配置 =====
# Run from the repository root or this notebook's directory.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'environment.yml').is_file() else Path.cwd().parent
BASE_DIR = PROJECT_ROOT / '01_preprocessing'
MOL2_ROOT = BASE_DIR / 'mol2'
EXCEL_PATH = BASE_DIR / 'experiments.xlsx'
OUTPUT_PT = PROJECT_ROOT / '02_datasets' / 'kare_gat_dataset.pt'

MOLECULE_DIR_NAME_ALIASES = {
    '': '',
}

print('mol2 根目录:', MOL2_ROOT)
print('Excel 文件:', EXCEL_PATH)
print('输出 pt 文件:', OUTPUT_PT)

In [ ]:
# 读取 Excel。
# A 列是分子名；B-M 列是实验条件；N 列是萃取效率 E%。
# I 列 “萃取的稀土金属” 与 J 列 “萃取的稀土金属原子序号” 等效，
# 因此实验条件中排除 I 列，只保留 J 列原子序号。
# 同时按当前建模需求排除有机相溶剂/改性剂 MPI 和体积比四列。
df = pd.read_excel(EXCEL_PATH, sheet_name=0)

molecule_col = df.columns[0]
excluded_condition_cols = [
    '萃取的稀土金属',
    '有机相溶剂MPI(eV)',
    '有机相改性剂MPI(eV)',
    '有机相溶剂体积比',
    '有机相改性剂体积比',
]
condition_cols = [col for col in df.columns[1:13] if col not in excluded_condition_cols]
extraction_rate_col = df.columns[13]

print('Excel shape:', df.shape)
print('分子名列:', molecule_col)
print('实验条件列 B-M，已排除不用于建模的列:')
for col in condition_cols:
    print('  -', col)
print('萃取率列 N:', extraction_rate_col)
print('分子种类数:', df[molecule_col].nunique())

In [ ]:
# I 列 “萃取的稀土金属” 已被排除，保留的实验条件列应全部为数值列。
# 这里仍保留数值列筛选和非数值列检查，便于后续 Excel 结构变化时及时发现问题。
numeric_condition_cols = [
    col for col in condition_cols
    if pd.api.types.is_numeric_dtype(df[col])
]
non_numeric_condition_cols = [col for col in condition_cols if col not in numeric_condition_cols]

print('数值型实验条件列，可进入 experiment_condition_tensor:')
for col in numeric_condition_cols:
    print('  -', col)

print('非数值实验条件列，当前不应存在；如存在，请检查 Excel 或继续排除/编码:')
for col in non_numeric_condition_cols:
    print('  -', col)

In [ ]:
def clean_scalar(value):
    """把 pandas/numpy 标量转换为更容易 torch.save 和人工审查的 Python 标量。"""
    if pd.isna(value):
        return None
    if hasattr(value, 'item'):
        return value.item()
    return value


def read_resp_charges_without_h(chg_path):
    """读取 .chg 文件最后一列 RESP 电荷，并跳过 H 原子。

    .chg 文件每行示例：
    C    -1.287813   -2.024835   -1.179317   0.0639843068

    本数据集中第 1 列是元素符号，最后一列是 RESP 电荷。由于 RDKit 读取 mol2 时设置
    removeHs=True，节点只保留非 H 原子，所以这里同步跳过 H。
    """
    charges = []
    with open(chg_path, 'r', encoding='utf-8') as f:
        for line_number, line in enumerate(f, start=1):
            parts = line.split()
            if not parts:
                continue
            element = parts[0]
            if element == 'H':
                continue
            try:
                charges.append(float(parts[-1]))
            except ValueError as exc:
                raise ValueError(f'{chg_path} 第 {line_number} 行最后一列无法转换为 float: {line!r}') from exc
    return charges


def molecule_name_from_dir(dir_name):
    """把 mol2 父目录名规范化为 Excel 中使用的分子名。"""
    return MOLECULE_DIR_NAME_ALIASES.get(dir_name, dir_name)


def conformer_id_from_path(mol2_path, molecule_name):
    """从文件名中提取构象标识。

    分子名可能包含连字符，例如 CA-12、WHT-2a。这里不按最后一个连字符硬切，
    而是优先删除父目录分子名前缀：
    WHT-2a-642-b3lyp.mol2 -> 642-b3lyp
    CA-12-001.mol2 -> 001
    """
    stem = mol2_path.stem
    candidate_prefixes = [molecule_name, mol2_path.parent.name]
    for prefix_name in candidate_prefixes:
        prefix = prefix_name + '-'
        if stem.startswith(prefix):
            return stem[len(prefix):]
    return stem


def mol2_chg_to_graph_data(mol2_path):
    """把单个 mol2/chg 构象转换为只包含结构图的 PyG Data。"""
    mol2_path = Path(mol2_path)
    chg_path = mol2_path.with_suffix('.chg')
    molecule_name = molecule_name_from_dir(mol2_path.parent.name)

    if not chg_path.exists():
        raise FileNotFoundError(f'找不到与 mol2 同名的 chg 文件: {chg_path}')

    # removeHs=True 是本任务要求：隐藏 H 原子。
    mol = MolFromMol2File(str(mol2_path), removeHs=True)
    if mol is None:
        raise ValueError(f'RDKit 无法读取 mol2 文件: {mol2_path}')

    charges = read_resp_charges_without_h(chg_path)
    atoms = list(mol.GetAtoms())
    if len(charges) != len(atoms):
        raise ValueError(
            f'非 H 原子数量与 RESP 电荷数量不一致: {mol2_path} | '
            f'RDKit atoms={len(atoms)}, chg charges={len(charges)}'
        )

    features = [
        [float(atom.GetAtomicNum()), float(charges[i])]
        for i, atom in enumerate(atoms)
    ]
    x = torch.tensor(features, dtype=torch.float32)

    adjacency = Chem.GetAdjacencyMatrix(mol)
    row, col = adjacency.nonzero()
    edge_index = torch.tensor([row, col], dtype=torch.long)

    return Data(
        x=x,
        atom_types_chg=x,
        edge_index=edge_index,
        name=molecule_name,
        conformer_id=conformer_id_from_path(mol2_path, molecule_name),
        mol2_path=str(mol2_path),
        chg_path=str(chg_path),
    )

In [ ]:
# 扫描所有 mol2 文件，并转换为结构图 Data。
# 文件组织为 mol2/<分子名>/<分子名>-<构象编号>.mol2，
# 所以分子名使用父目录名，能正确处理 CA-12、WHT-2a 这类名字。
mol2_files = sorted(MOL2_ROOT.glob('*/*.mol2'))
print('发现 mol2 文件数:', len(mol2_files))

ligand_graphs = []
errors = []
for mol2_path in mol2_files:
    try:
        ligand_graphs.append(mol2_chg_to_graph_data(mol2_path))
    except Exception as exc:
        errors.append((str(mol2_path), repr(exc)))

print('成功转换结构图数:', len(ligand_graphs))
print('转换失败数:', len(errors))
if errors:
    for path, err in errors[:20]:
        print(path, err)
    raise RuntimeError('存在 mol2/chg 转换失败，请先检查上面列出的文件。')

In [ ]:
# 按分子名汇总构象图，并检查 Excel 分子名是否都有对应 mol2 构象。
graphs_by_molecule = defaultdict(list)
for graph in ligand_graphs:
    graphs_by_molecule[graph.name].append(graph)

excel_molecules = set(df[molecule_col].astype(str))
graph_molecules = set(graphs_by_molecule)

missing_in_graphs = sorted(excel_molecules - graph_molecules)
extra_graph_molecules = sorted(graph_molecules - excel_molecules)

print('mol2/chg 中分子种类数:', len(graph_molecules))
print('Excel 中有、mol2/chg 中没有的分子:', missing_in_graphs)
print('mol2/chg 中有、Excel 中没有的分子:', extra_graph_molecules)

if missing_in_graphs:
    raise RuntimeError('Excel 中存在没有 mol2/chg 构象的分子，请检查分子名或文件目录。')

In [ ]:
# 将结构图与 Excel 实验行按 molecule_name 做交叉连接。
# 对每个 Excel 行，复制该分子的每个构象图，并附加实验条件和萃取率。
data_list = []

for excel_row_index, row in df.iterrows():
    molecule_name = str(row[molecule_col])
    condition_dict = {col: clean_scalar(row[col]) for col in condition_cols}
    condition_tensor_values = [clean_scalar(row[col]) for col in numeric_condition_cols]
    condition_tensor = torch.tensor([condition_tensor_values], dtype=torch.float32)
    extraction_rate = torch.tensor([[float(row[extraction_rate_col])]], dtype=torch.float32)

    for graph in graphs_by_molecule[molecule_name]:
        data = Data(
            x=graph.x.clone(),
            atom_types_chg=graph.atom_types_chg.clone(),
            edge_index=graph.edge_index.clone(),
            experiment_condition=condition_dict,
            experiment_condition_tensor=condition_tensor.clone(),
            extraction_rate=extraction_rate.clone(),
            name=molecule_name,
            conformer_id=graph.conformer_id,
            mol2_path=graph.mol2_path,
            chg_path=graph.chg_path,
            excel_row_index=int(excel_row_index),
        )
        data_list.append(data)

print('最终 Data 对象数量:', len(data_list))
expected_count = sum(len(graphs_by_molecule[str(row[molecule_col])]) for _, row in df.iterrows())
print('按交叉连接计算的预期数量:', expected_count)
assert len(data_list) == expected_count

In [ ]:
# 审查一个样本，确认字段、形状和值类型符合预期。
sample = data_list[0]
print(sample)
print('x shape:', tuple(sample.x.shape), 'dtype:', sample.x.dtype)
print('edge_index shape:', tuple(sample.edge_index.shape), 'dtype:', sample.edge_index.dtype)
print('experiment_condition keys:', list(sample.experiment_condition.keys()))
print('experiment_condition:', sample.experiment_condition)
print('experiment_condition_tensor shape:', tuple(sample.experiment_condition_tensor.shape))
print('extraction_rate:', sample.extraction_rate)
print('name:', sample.name)
print('conformer_id:', sample.conformer_id)
print('mol2_path:', sample.mol2_path)

In [ ]:
# 统计每个分子的构象数、Excel 行数和最终样本数，便于人工审查。
summary_rows = []
for molecule_name in sorted(excel_molecules):
    num_conformers = len(graphs_by_molecule[molecule_name])
    num_excel_rows = int((df[molecule_col].astype(str) == molecule_name).sum())
    summary_rows.append({
        '分子名': molecule_name,
        '构象数': num_conformers,
        'Excel行数': num_excel_rows,
        '交叉连接后样本数': num_conformers * num_excel_rows,
    })

summary_df = pd.DataFrame(summary_rows)
print('交叉连接后总样本数:', int(summary_df['交叉连接后样本数'].sum()))
summary_df

In [ ]:
# 保存数据集。
# 加载方式示例：loaded_data_list = torch.load(OUTPUT_PT)
OUTPUT_PT.parent.mkdir(parents=True, exist_ok=True)
torch.save(data_list, OUTPUT_PT)
print('已保存:', OUTPUT_PT)
print('保存的 Data 对象数量:', len(data_list))

In [ ]:
# 可选：立即读回保存文件，做一次最基本的完整性验证。
loaded_data_list = torch.load(OUTPUT_PT)
print('读回 Data 对象数量:', len(loaded_data_list))
assert len(loaded_data_list) == len(data_list)
assert torch.equal(loaded_data_list[0].x, data_list[0].x)
assert torch.equal(loaded_data_list[0].edge_index, data_list[0].edge_index)
assert torch.equal(loaded_data_list[0].extraction_rate, data_list[0].extraction_rate)
print('读回验证通过。')